In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

1. Data Loading
We load the preprocessed dataset. Multinomial Naive Bayes requires non-negative feature values, which aligns perfectly with the output of the TF-IDF vectorizer.

In [ ]:
INPUT_PATH = "../data/preprocessed/cleaned_reviews.csv"
OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)
df = pd.read_csv(INPUT_PATH)
df['cleaned_review'] = df['cleaned_review'].fillna('')

2. Feature Extraction (TF-IDF)
We transform the text data into numerical format using "TF-IDF". We maintain the same `max_features` as the Random Forest model to ensure a fair comparison between the two classifiers.

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['cleaned_review'])
y = df['label_binary']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

3. Model Training: Naive Bayes
We initialize the "MultinomialNB" classifier. This model calculates the probability of each word belonging to a specific class (Fake or Real) and uses Bayes' Theorem to make the final prediction.

In [ ]:
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

### 4. Performance Evaluation
The performance is measured using **Accuracy** and visualized via a **Confusion Matrix**. Naive Bayes often provides a strong baseline due to its computational efficiency.

In [ ]:
y_pred = nb_model.predict(X_test)
print(f" Naive Bayes Accuracy: {accuracy_score(y_test, y_pred):.4f}")
plt.figure(figsize=(8, 6))
sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Oranges',
 xticklabels=['Fake', 'Real'], yticklabels=['Fake', 'Real'])
plt.title('Naive Bayes - Confusion Matrix')
plt.show()

5. Saving the Model
The trained Naive Bayes model and its corresponding vectorizer are saved as `.pkl` files for future deployment and integration.

In [ ]:
joblib.dump(nb_model, os.path.join(OUTPUT_DIR, "nb_model.pkl"))
joblib.dump(vectorizer, os.path.join(OUTPUT_DIR, "nb_tfidf_vectorizer.pkl"))